Modele tek seferde girebilecek en fazla token sayısı -> context length
bu bir üst sınır, her seferinde o kadar token almak zorunda değil
mesela modelimizin context length'i 32k, 50 tokenlık girdi verirsek sadece 50 token işler, geri kalanı pad ile doldurulmaz
cevap da tek seferde gelmez, model her adımda 1 token üretir, onu girdiye ekler, tekrar üretir
(girdi + cevap toplamı 32k'yı geçemez)
pad sadece eğitimde farklı uzunluktaki cümleleri aynı boya getirmek için kullanılır

eğitimde "the capital of amerika dc" verdiğimizde
girdi: "the capital of amerika"
çıktı (hedef): "capital of amerika dc"
yani hedef, girdinin 1 kaydırılmış hali
her kelime kendinden sonrakini tahmin ediyor:
the -> capital, capital -> of, of -> amerika, amerika -> dc
yani model aslında sonraki tokenı tahmin ediyor, sadece hepsini aynı anda yapıyor

In [32]:
context_length = 8

train setinde: 
    girdi = "the capital of amerika dc"
    çıktı = "capital of amerika dc states"
şeklinde olucak bu şekilde model bir sonraki tokenın ne olması gerketiğini öğreniyor

Bu adımıda pytorch ile yapacağız.

In [33]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("data/hf_tokanizer.json")

with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

token_ids = tokenizer.encode(text).ids

token_ids

[12,
 52,
 32,
 40,
 38,
 36,
 35,
 30,
 32,
 4,
 26,
 52,
 32,
 40,
 38,
 30,
 32,
 43,
 48,
 32,
 48,
 35,
 48,
 35,
 61,
 22,
 32,
 50,
 34,
 50,
 35,
 50,
 35,
 32,
 36,
 35,
 40,
 38,
 36,
 33,
 33,
 48,
 51,
 26,
 32,
 30,
 33,
 54,
 43,
 47,
 35,
 52,
 35,
 25,
 53,
 30,
 33,
 34,
 26,
 39,
 30,
 43,
 52,
 46,
 22,
 33,
 50,
 51,
 57,
 54,
 42,
 38,
 26,
 52,
 38,
 55,
 61,
 39,
 30,
 39,
 40,
 26,
 34,
 52,
 38,
 55,
 23,
 30,
 33,
 30,
 34,
 30,
 25,
 53,
 6,
 9,
 53,
 54,
 42,
 38,
 60,
 55,
 40,
 26,
 34,
 26,
 33,
 43,
 22,
 37,
 50,
 40,
 22,
 51,
 62,
 50,
 25,
 53,
 60,
 46,
 4,
 32,
 36,
 35,
 25,
 57,
 39,
 22,
 40,
 47,
 38,
 4,
 23,
 36,
 23,
 55,
 4,
 25,
 30,
 43,
 36,
 40,
 61,
 40,
 38,
 57,
 39,
 30,
 39,
 40,
 47,
 38,
 25,
 48,
 38,
 6,
 11,
 53,
 60,
 46,
 22,
 32,
 50,
 34,
 22,
 32,
 56,
 51,
 50,
 32,
 36,
 43,
 56,
 61,
 17,
 29,
 34,
 15,
 57,
 41,
 35,
 41,
 1,
 35,
 22,
 28,
 47,
 38,
 26,
 48,
 44,
 58,
 55,
 54,
 35,
 28,
 26,
 46,
 60,
 22,
 32,
 50

In [34]:
ids_text = ""

for token_id in token_ids:
    ids_text += f"{token_ids} "

with open("data/data_toke_ids.txt" ,"w") as f:
    f.write(ids_text)

In [35]:
tokenizer.add_special_tokens(["<pad>"])
tokenizer.save("data/hf_tokanizer.json")

In [36]:
import torch
from torch.utils.data import DataLoader, Dataset

pad_id = 64

class TextDataset(Dataset):
    def __init__(self, token_ids: list, context_length: int, stride: int):
        super().__init__()

        self.input = []
        self.target = []

        for i in range(0,len(token_ids)- context_length, stride):
            input_chunk = token_ids[i: i+context_length]
            output_chunk = token_ids[i+1 : i + 1 + context_length]

            #bir şekilde fazla gelirse traşla
            input_chunk = input_chunk[:context_length]
            output_chunk = output_chunk[:context_length]

            #eksik gelirse pad ile doldur
            input_chunk = input_chunk + [pad_id] * (context_length - len(input_chunk))
            output_chunk = output_chunk + [pad_id] * (context_length - len(output_chunk))

            #fazla dolarsa tekrar traşla
            input_chunk = input_chunk[:context_length]
            output_chunk = output_chunk[:context_length]

            self.input.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target.append(torch.tensor(output_chunk, dtype=torch.long))


    def __len__(self):
        return len(self.input)

    def __getitem__(self, index):
        return self.input[index], self.target[index]


In [37]:
def create_data_loader(token_ids: list, context_length: int, stride:int, batch_size: int, 
                       shuffle: bool = True, device:str= "cpu"):
    dataset = TextDataset(token_ids, context_length, stride)
    dataloader=DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=torch.Generator(device=device)
    )
    return dataloader

In [38]:
train_data_loader = create_data_loader(token_ids, context_length, 1, 1)
len(train_data_loader)

1477